In [1]:
import os
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
import gplately
import pygplates
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from plate_model_manager import PlateModelManager

In [2]:
# Download Muller et al. 2019 files
pm_manager = PlateModelManager()
plate_model = pm_manager.get_model("Zahirovic2022", data_dir="plate-model-repo")

rotation_model = plate_model.get_rotation_model()
topology_features = plate_model.get_topologies()
static_polygons = plate_model.get_static_polygons()

In [3]:
# Obtain geometries 
coastlines = plate_model.get_layer('Coastlines')
continents = plate_model.get_layer('ContinentalPolygons')
#COBs =  plate_model.get_layer('COBs')

# Call the PlotTopologies object
model = gplately.PlateReconstruction(rotation_model, topology_features, static_polygons)
gplot = gplately.PlotTopologies(model, coastlines, continents)

# Create a TopologicalModel from the topologies and rotation model.
topological_model = pygplates.TopologicalModel(topology_features, rotation_model)

In [4]:
def calculate_convergence_velocities(topological_snapshot):

    # Calculate statistics along plate boundary sections.
    plate_boundary_stats = topological_snapshot.calculate_plate_boundary_statistics(
            np.radians(2),  # 2 degree spacing between points
            first_uniform_point_spacing_radians=0.0,
            velocity_units=pygplates.VelocityUnits.cms_per_yr)
    
    # Record boundary points and their convergence velocities (along the uniformly sampled subduction zones).
    boundary_stats = []
    
    # Iterate over the uniformly sampled points along plate boundary.
    for stat in plate_boundary_stats:
        if (True or stat.left_plate.located_in_resolved_network() or
            stat.right_plate.located_in_resolved_network()):
            if stat.convergence_velocity:
                boundary_stats.append(stat)
    
    north_east_down_convergence_velocities = pygplates.LocalCartesian.convert_from_geocentric_to_north_east_down(
            [stat.boundary_point for stat in boundary_stats],
            [stat.convergence_velocity for stat in boundary_stats])

    # Convert pygplates.PointOnSphere to (lon, lat).
    # And convert pygplates.Vector3D to (east, north) and magnitude.
    lons = np.empty(len(boundary_stats))
    lats = np.empty(len(boundary_stats))
    conv_vel_x = np.empty(len(boundary_stats))
    conv_vel_y = np.empty(len(boundary_stats))
    conv_vel_signed_mag = np.empty(len(boundary_stats))
    for index, stat in enumerate(boundary_stats):
        lat, lon = stat.boundary_point.to_lat_lon()
        lons[index] = lon
        lats[index] = lat
        conv_vel_x[index] = north_east_down_convergence_velocities[index].get_y()  # east
        conv_vel_y[index] = north_east_down_convergence_velocities[index].get_x()  # north
        conv_vel_signed_mag[index] = stat.convergence_velocity_signed_magnitude

    return  lons, lats, conv_vel_x, conv_vel_y, conv_vel_signed_mag

In [5]:
    def subdivide_network_triangle(triangle, triangles, vertex_points, vertex_dilatation_rates, resolved_network, subdivide_threshold):
        vert_index1, vert_index2, vert_index3 = triangle

        # Get dot-product distances between triangle's vertices.
        point1, point2, point3 = vertex_points[vert_index1], vertex_points[vert_index2], vertex_points[vert_index3]
        dot12 = pygplates.Vector3D.dot(point1.to_xyz(), point2.to_xyz())
        dot13 = pygplates.Vector3D.dot(point1.to_xyz(), point3.to_xyz())
        dot23 = pygplates.Vector3D.dot(point2.to_xyz(), point3.to_xyz())
        
    	# If no need to further subdivide then add triangle and return.
        if (dot12 > subdivide_threshold and
            dot13 > subdivide_threshold and
            dot23 > subdivide_threshold):
            triangles.append(triangle)
            return
        
    	# Subdivide into two triangles by splitting the longest edge of current triangle.
        if dot12 < dot13:
            if dot12 < dot23:
                point12 = pygplates.GreatCircleArc(point1, point2).get_arc_point(0.5)  # mid-point
                strain_rate = resolved_network.get_point_strain_rate(point12)
                if strain_rate is None:
                    dilatation_rate = 0.0
                else:
                    dilatation_rate = strain_rate.get_dilatation_rate()
                
                vert_index12 = len(vertex_points)
                vertex_points.append(point12)
                vertex_dilatation_rates.append(dilatation_rate)
                
                subdivide_network_triangle(
                        (vert_index1, vert_index12, vert_index3),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
                subdivide_network_triangle(
                        (vert_index12, vert_index2, vert_index3),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
            else:
                point23 = pygplates.GreatCircleArc(point2, point3).get_arc_point(0.5)  # mid-point
                strain_rate = resolved_network.get_point_strain_rate(point23)
                if strain_rate is None:
                    dilatation_rate = 0.0
                else:
                    dilatation_rate = strain_rate.get_dilatation_rate()
                
                vert_index23 = len(vertex_points)
                vertex_points.append(point23)
                vertex_dilatation_rates.append(dilatation_rate)
                
                subdivide_network_triangle(
                        (vert_index1, vert_index2, vert_index23),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
                subdivide_network_triangle(
                        (vert_index1, vert_index23, vert_index3),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
        else:
            if dot13 < dot23:
                point13 = pygplates.GreatCircleArc(point1, point3).get_arc_point(0.5)  # mid-point
                strain_rate = resolved_network.get_point_strain_rate(point13)
                if strain_rate is None:
                    dilatation_rate = 0.0
                else:
                    dilatation_rate = strain_rate.get_dilatation_rate()
                
                vert_index13 = len(vertex_points)
                vertex_points.append(point13)
                vertex_dilatation_rates.append(dilatation_rate)
                
                subdivide_network_triangle(
                        (vert_index1, vert_index2, vert_index13),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
                subdivide_network_triangle(
                        (vert_index13, vert_index2, vert_index3),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
            else:
                point23 = pygplates.GreatCircleArc(point2, point3).get_arc_point(0.5)  # mid-point
                strain_rate = resolved_network.get_point_strain_rate(point23)
                if strain_rate is None:
                    dilatation_rate = 0.0
                else:
                    dilatation_rate = strain_rate.get_dilatation_rate()
                
                vert_index23 = len(vertex_points)
                vertex_points.append(point23)
                vertex_dilatation_rates.append(dilatation_rate)
                
                subdivide_network_triangle(
                        (vert_index1, vert_index2, vert_index23),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)
                subdivide_network_triangle(
                        (vert_index1, vert_index23, vert_index3),
                        triangles, vertex_points, vertex_dilatation_rates,
                        resolved_network, subdivide_threshold)

    def subdivide_network_triangulations(topological_snapshot, subdivide_degrees, max_dilatation_rate=500*1e-17):
        triangulations = []

        subdivide_threshold = np.cos(np.radians(subdivide_degrees))
        
        resolved_networks = topological_snapshot.get_resolved_topologies(pygplates.ResolveTopologyType.network)
        for resolved_network in resolved_networks:
            network_triangulation = resolved_network.get_network_triangulation()
            network_vertices = network_triangulation.get_vertices()

            # Un-subdivided vertex locations and strain rates.
            vertex_points = []
            vertex_dilatation_rates = []
            for vertex in network_vertices:
                vertex_points.append(vertex.position)
                vertex_dilatation_rates.append(vertex.strain_rate.get_dilatation_rate())
            
            # Map each un-subdivided vertex to its vertex index.
            network_vertex_indices = dict((network_vertices[index], index) for index in range(len(network_vertices)))

            # Subdivide each triangle *in the deforming region*.
            triangles = []
            for network_triangle in network_triangulation.get_triangles():
                if network_triangle.is_in_deforming_region:
                    # Get this triangle's 3 vertex indices.
                    triangle = tuple(network_vertex_indices[network_triangle.get_vertex(vert_index)] for vert_index in (0,1,2))
                    subdivide_network_triangle(triangle, triangles, vertex_points, vertex_dilatation_rates, resolved_network, subdivide_threshold)

            if triangles:
                vertex_lons = np.empty(len(vertex_points))
                vertex_lats = np.empty(len(vertex_points))
                for index in range(len(vertex_points)):
                    lat, lon = vertex_points[index].to_lat_lon()
                    vertex_lons[index] = lon
                    vertex_lats[index] = lat
                vertex_dilatation_rates = np.array(vertex_dilatation_rates)
                
                triangulation = triangles, vertex_lons, vertex_lats, vertex_dilatation_rates
                triangulations.append(triangulation)

        return triangulations

In [6]:
# A function to plot latitudinal labels
def latlonticks(ax):
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
              linewidth=1, color='gray', alpha=0.3,)
    
    gl.top_labels=False
    gl.bottom_labels=False

        
    # Set font size for labels
    gl.xlabel_style = {'size': 14}  # Font size for longitude labels
    gl.ylabel_style = {'size': 14}  # Font size for latitude labels

    return

In [7]:
def generate_frame(output_filename, time, lon_lat_extent=None):

    # Set up a GeoAxis plot
    fig = plt.figure(figsize=(24,18))
    ax = fig.add_subplot(111, projection=ccrs.Mercator(central_longitude = 0))
    #ax.gridlines(color='0.7',linestyle='--', xlocs=np.arange(-180,180,15), ylocs=np.arange(-90,90,15))
    plt.title('Plate boundary convergence/divergence and deforming region strain rates at %i Ma' % (time), fontsize=16)
    
    # Plot all topologies
    gplot.time=time
    #gplot.plot_continents(ax, facecolor='0.95')
    gplot.plot_coastlines(ax, facecolor='0.90', color='0.5')
    gplot.plot_misc_boundaries(ax, color='k', zorder=3)
    gplot.plot_ridges(ax, color='k', zorder=3)
    gplot.plot_transforms(ax, color='k', zorder=3)
    #gplot.plot_ridges_and_transforms(ax, color='k', zorder=3)
    gplot.plot_trenches(ax, color='w', linewidth=10)
    gplot.plot_trenches(ax, color='k', zorder=3)
    gplot.plot_subduction_teeth(ax, spacing=np.radians(1), color='k', zorder=3)

    # Get a snapshot of our resolved topologies at the current 'time'.
    topological_snapshot = topological_model.topological_snapshot(time)


    def draw_network_triangulation(max_dilatation_rate=500*1e-17):
        im = None
        
        triangulations = subdivide_network_triangulations(topological_snapshot, subdivide_degrees=0.5)
        for triangles, vertex_lons, vertex_lats, vertex_dilatation_rates in triangulations:
            # Plot the triangulation.
            mtri = mpl.tri.Triangulation(vertex_lons, vertex_lats, triangles)
            #ax.triplot(triangulation, 'ko-', transform=ccrs.PlateCarree())
            im = ax.tripcolor(
                    mtri,
                    1e17 * vertex_dilatation_rates,  # in units of 10^(-17)
                    transform=ccrs.PlateCarree(),
                    # Seems 'gouraud' doesn't plot anything...
                    shading='flat',
                    cmap=plt.cm.coolwarm,  # plt.cm.bwr,
                    norm=mpl.colors.SymLogNorm(linthresh=0.01, vmin=-1e17*max_dilatation_rate, vmax=1e17*max_dilatation_rate))

        if im:
            # Add colorbar (in units of 10^(-17)).
            cax1 = fig.add_axes([0.9, 0.35, 0.02, 0.5])
            fig.colorbar(im, cax=cax1, shrink=0.5, extend='both').set_label(r'Dilitation rate $(10^{-17}\,\frac{1}{s})$', fontsize=14)

    # Draw the triangulation of each deforming network in the topological snapshot.
    draw_network_triangulation()

    # Draw the point locations.
    #
    # Update: Not drawing because Muller2019 model has networks 'adjoining' rigid plates (not overlapping them).
    #         And so, in the Andes for example, we get points 'inside' the SAM plate (not just at trench).
    #ax.scatter(lons, lats, transform=ccrs.PlateCarree(), c=conv_vel_signed_mag, s=10, cmap=plt.cm.vanimo, vmin=-10, vmax=10, zorder=2)
    
    # Plot the convergence velocity arrows (with their velocity magnitudes as a colour scale).
    def draw_convergence_velocity(topological_snapshot, invert_arrow_directions, scale=300):
        
        lons, lats, conv_vel_x, conv_vel_y, conv_vel_signed_mag = calculate_convergence_velocities(topological_snapshot)
        
        if invert_arrow_directions:
            conv_vel_x = -conv_vel_x
            conv_vel_y = -conv_vel_y
        
        return ax.quiver(
                lons,
                lats,
                conv_vel_x,
                conv_vel_y,
                conv_vel_signed_mag,
                scale=scale,
                transform=ccrs.PlateCarree(),
                cmap=plt.cm.PRGn_r,
                clim=(-10,10),
                regrid_shape=None,
                zorder=2)

    # For each point draw two arrows in opposite directions.
    # This is because could be converging or diverging, and we use colour map to distinguish (not arrow direction).
    # TODO: Might be possible to draw 'converging' velocities as two 'inward' pointing arrows (and use 'outward' for diverging).
    #       Not sure if something simple like the following would work? ...
    #       https://stackoverflow.com/questions/59265233/how-to-invert-the-direction-of-arrows-from-radially-outwards-to-radially-inwards
    im = draw_convergence_velocity(topological_snapshot, invert_arrow_directions=False)
    im = draw_convergence_velocity(topological_snapshot, invert_arrow_directions=True)

    # Add colorbar.
    #fig.colorbar(im, ax=ax, shrink=0.5, extend='both').set_label('Convergence/divergence velocity magntitude (cm/yr)', fontsize=14)

    cax2 = fig.add_axes([0.78, 0.35, 0.02, 0.5]) 
    fig.colorbar(im, cax=cax2, shrink=0.5, extend='both').set_label('Convergence/divergence velocity magntitude (cm/yr)', fontsize=14)

    fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                    wspace=0.001, hspace=0.3)

    latlonticks(ax)
    if lon_lat_extent is None:
        ax.set_global()
    else:
        ax.set_extent(lon_lat_extent, crs=ccrs.PlateCarree())

    plt.savefig(output_filename, bbox_inches='tight', dpi=300)
    plt.close()

In [8]:
lon_lat_extent = [65, 135, -10, 50]
time_range = np.arange(40,-1,-1)

nprocs = 8

parallel = None
if nprocs != 1:
    try:
        from joblib import Parallel

        parallel = Parallel(nprocs)
    except ImportError:
        print("Could not import joblib; falling back to serial execution")

if parallel is None:
    for time in time_range:
        print('Generating %d Ma frame...' % time)
        generate_frame("./release_vid_snapshot_{}.png".format(time), time, lon_lat_extent)
else:
    from joblib import delayed

    parallel(
        delayed(generate_frame)(
            output_filename="./release_vid_snapshot_{}.png".format(time),
            time=time,
            lon_lat_extent=lon_lat_extent,
        )
        for time in time_range
    )

In [11]:
try:
    import moviepy.editor as mpy  # moviepy 1.x
except ImportError:
    import moviepy as mpy  # moviepy 2.x

frame_list = []
for time in time_range:
    frame_list.append(
        "./release_vid_snapshot_{}.png".format(time)
    )
    
clip = mpy.ImageSequenceClip(frame_list, fps=10)

clip.write_videofile(
    "./pygplates_release_video.mp4",
    fps=15,
    codec="libx264",
    bitrate="8000k",
    audio=False,
    logger=None,
    ffmpeg_params=[
        "-vf",
        "pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-pix_fmt",
        "yuv420p",
    ],
)

#from IPython.display import Image
#
#clip.write_gif("./pygplates_release_video.gif")
#
#with open("./pygplates_release_video.gif", "rb") as f:
#    display(Image(data=f.read(), format='gif', width = 3000, height = 1000))